# NIDS -- Network Intrusion Detection System
## CNN-BiLSTM Model Training Playbook

**Version** 3.0 &nbsp;|&nbsp; **Last updated** 2026-03-10

---

### Table of Contents

| # | Section | Purpose |
|---|---------|----------|
| 1 | [Setup](#1) | Install deps, imports, GPU, seeds |
| 2 | [Configuration](#2) | All hyperparameters in one place |
| 3 | [Data Loading](#3) | Load preprocessed arrays + metadata |
| 4 | [Exploratory Data Analysis](#4) | Class balance, feature stats, correlations |
| 5 | [Model Architecture](#5) | CNN-BiLSTM definition + sanity check |
| 6 | [DataLoaders & Sampling](#6) | WeightedRandomSampler for imbalanced data |
| 7 | [Training](#7) | AMP training loop, early stopping, LR schedule |
| 8 | [Training Curves](#8) | Loss, accuracy, F1, LR plots |
| 9 | [Evaluation](#9) | Test metrics, confusion matrix, ROC, error analysis |
| 10 | [ONNX Export](#10) | Export, verify equivalence, benchmark |
| 11 | [Summary & Artifacts](#11) | Save results, list output files |

---

### Task

Train a **CNN-BiLSTM** hybrid classifier on **77 bidirectional flow-level
features** extracted by the C++ `NativeFlowExtractor` to discriminate 16 traffic
classes (1 benign + 15 attack types) in real time.

### Dataset

**LSNM2024** -- Wireshark packet captures preprocessed into ~992K bidirectional
flows by `scripts/ml/preprocess.py` (with max-flow splitting at 200 packets).

> Lashkari, A.H.; Sabzevar, H.H.; Arash, H.R. (2024).
> *LSNM2024 -- A Labeled and Structured Network Traffic Dataset for ML-based
> Intrusion Detection*. Mendeley Data, V1.
> [doi:10.17632/sdnfh32352.1](https://doi.org/10.17632/sdnfh32352.1)

### Prerequisites

| Requirement | Detail |
|-------------|--------|
| Runtime | Google Colab with **GPU** (T4 / V100 / A100) or local GPU |
| Data | 7 `.npy` files from `scripts/ml/preprocess.py` (~310 MB) |
| Disk | ~1 GB free (model + checkpoints + plots) |
| Time | ~15-30 min on a T4 GPU |

<a id="1"></a>

---
## 1. Setup

In [ ]:
import subprocess, sys

_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in __import__("os").environ

_install = [
    "onnx==1.17.0", "onnxruntime==1.21.0", "onnxscript",
    "seaborn==0.13.2", "tqdm==4.67.1",
    "pandas==2.2.2" if _COLAB else "pandas==2.2.3",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"]
    + ([] if _COLAB else ["--user"])
    + _install
)
print("Dependencies OK.")

In [ ]:
# ---- All imports (consolidated) ----
import copy
import hashlib
import json
import os
import random
import shutil
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300,
                     "savefig.bbox": "tight", "font.size": 10})
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda or 'N/A'}  |  Device: {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  ({props.total_memory / 1e9:.1f} GB)")
else:
    print("WARNING: No GPU detected -- training will be very slow.")

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Seed: {SEED}  |  Deterministic: {torch.backends.cudnn.deterministic}")

<a id="2"></a>

---
## 2. Configuration

**Every tunable value lives here.** No magic numbers elsewhere in the notebook.

| Group | Parameters | Rationale |
|-------|------------|-----------|
| Architecture | CNN filters, kernel, LSTM hidden, dense widths, dropouts | Matches the C++ `OnnxAnalyzer` contract |
| Optimiser | AdamW, lr 1e-3, weight_decay 1e-4 | Decoupled weight decay (Loshchilov & Hutter 2019) |
| Scheduler | ReduceLROnPlateau, factor 0.5, patience 3 | Halves LR when val loss stalls |
| Sampling | Inverse-frequency weights, 50x-median cap | Prevents majority-class dominance; reduced from 100x to curb false positives |
| Loss | CrossEntropyLoss + label_smoothing 0.05 | Reduced from 0.1 to preserve recall on rare classes |
| Training | 100 epochs, patience 10, grad clip 1.0, AMP | Wider window for convergence; AMP ~doubles throughput on T4 |

In [ ]:
# ===================== Hyperparameters =====================
# -- Architecture --
CNN_CHANNELS      = (64, 128)       # Conv1d output channels
CNN_KERNEL        = 3               # Conv1d kernel size (with padding=1)
LSTM_HIDDEN       = 128             # BiLSTM hidden size (output = 2x)
DENSE_WIDTHS      = (256, 128)      # FC layer widths before logit head
DROP_CNN          = 0.2             # Dropout after each CNN block
DROP_LSTM         = 0.3             # Dropout after BiLSTM
DROP_DENSE        = (0.4, 0.3)      # Dropout in dense block

# -- Optimiser & schedule --
BATCH_SIZE        = 512
LEARNING_RATE     = 1e-3
WEIGHT_DECAY      = 1e-4
LABEL_SMOOTHING   = 0.05            # reduced from 0.1 — less confidence penalty
LR_FACTOR         = 0.5             # ReduceLROnPlateau reduction factor
LR_PATIENCE       = 3               # epochs before LR reduction
GRAD_CLIP_NORM    = 1.0

# -- Training --
MAX_EPOCHS        = 100             # doubled — previous run hit epoch limit
PATIENCE          = 10              # early-stopping patience (wider window)
USE_AMP           = torch.cuda.is_available()

# -- Sampling --
SAMPLER_CAP_RATIO = 50.0            # reduced from 100 — less aggressive oversampling

# -- Output --
OUTPUT_DIR        = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Configuration OK.  Output directory:", OUTPUT_DIR.resolve())


In [ ]:
# ---- Stdout-to-file logging ----
# Mirrors all print output to a log file so you can download a complete
# training transcript instead of copy-pasting from Colab cells.

class _Tee:
    """Write to both the original stream and a log file."""

    def __init__(self, log_path: Path, stream):
        self._file = open(log_path, "w", buffering=1)  # line-buffered
        self._stream = stream

    def write(self, data: str) -> int:
        self._stream.write(data)
        self._file.write(data)
        return len(data)

    def flush(self):
        self._stream.flush()
        self._file.flush()

    def close(self):
        self._file.close()

    # Needed so tqdm / other libs treat this as a real file object
    @property
    def encoding(self):
        return getattr(self._stream, "encoding", "utf-8")

    def isatty(self) -> bool:
        return False

    def fileno(self):
        return self._stream.fileno()


_LOG_PATH = OUTPUT_DIR / "training_log.txt"
sys.stdout = _Tee(_LOG_PATH, sys.__stdout__)
sys.stderr = _Tee(OUTPUT_DIR / "training_log_err.txt", sys.__stderr__)

print(f"Logging stdout to: {_LOG_PATH.resolve()}")
print(f"Logging stderr to: {(OUTPUT_DIR / 'training_log_err.txt').resolve()}")

<a id="3"></a>

---
## 3. Data Loading

The preprocessing pipeline (`scripts/ml/preprocess.py`) produces 7 files:

- `X_train.npy`, `X_val.npy`, `X_test.npy` -- feature arrays (77 features)
- `y_train.npy`, `y_val.npy`, `y_test.npy` -- integer labels (0..15)
- `model_metadata.json` -- class names, feature names, normalisation params

The cell below searches common paths and falls back to extracting
`nids_processed_data.zip` if present.

In [ ]:
REQUIRED_FILES = [
    "X_train.npy", "X_val.npy", "X_test.npy",
    "y_train.npy", "y_val.npy", "y_test.npy",
    "model_metadata.json",
]


def find_data() -> Path:
    """Locate preprocessed data files or extract from zip."""
    for d in [Path("."), Path("data/processed"), Path("data_processed"),
              Path("scripts/data/processed"), Path("../scripts/data/processed")]:
        if all((d / f).exists() for f in REQUIRED_FILES):
            return d
    for zp in [Path("nids_processed_data.zip"),
               Path("scripts/data/nids_processed_data.zip"),
               Path("data/nids_processed_data.zip"),
               Path("../scripts/data/nids_processed_data.zip"),
               Path("/content/nids_processed_data.zip")]:
        if zp.exists():
            dest = Path("data_processed")
            dest.mkdir(exist_ok=True)
            print(f"Extracting {zp} ({zp.stat().st_size / 1e6:.1f} MB) ...")
            with zipfile.ZipFile(zp) as z:
                z.extractall(dest)
            return dest
    raise FileNotFoundError(
        "Preprocessed data not found. Run scripts/ml/preprocess.py first, "
        "or place nids_processed_data.zip in the working directory."
    )


DATA_DIR = find_data()

# ---- Load arrays + metadata ----
X_train = np.load(DATA_DIR / "X_train.npy")
X_val   = np.load(DATA_DIR / "X_val.npy")
X_test  = np.load(DATA_DIR / "X_test.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
y_val   = np.load(DATA_DIR / "y_val.npy")
y_test  = np.load(DATA_DIR / "y_test.npy")

with open(DATA_DIR / "model_metadata.json") as f:
    metadata = json.load(f)

n_features   = X_train.shape[1]
n_classes    = metadata["n_classes"]
class_names  = [metadata["index_to_label"][str(i)] for i in range(n_classes)]
feature_names = metadata.get("feature_names", [f"f{i}" for i in range(n_features)])
clip_value   = metadata.get("normalization", {}).get("clip_value", 10.0)

n_total = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]
mem_mb = (X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1e6

print(f"Data: {DATA_DIR.resolve()}")
print(f"Train: {X_train.shape[0]:>9,}  |  Val: {X_val.shape[0]:>9,}  |  Test: {X_test.shape[0]:>9,}  |  Total: {n_total:>9,}")
print(f"Features: {n_features}  |  Classes: {n_classes}  |  Clip: {clip_value}  |  Memory: {mem_mb:.0f} MB")

<a id="4"></a>

---
## 4. Exploratory Data Analysis

Before training we inspect:

1. **Class distribution** -- how severe is the imbalance?
2. **Split consistency** -- did stratification preserve proportions?
3. **Data quality** -- NaN / Inf / values beyond the clip threshold?
4. **Feature statistics** -- constant features, skewness, variance ranking
5. **Feature distributions** -- histograms of the most informative features
6. **Correlations** -- highly correlated pairs that the model may exploit

In [ ]:
# ---- 4.1  Class distribution across all splits ----
splits = {"Train": y_train, "Val": y_val, "Test": y_test}
rows = []
for split_name, y in splits.items():
    counts = np.bincount(y, minlength=n_classes)
    for i in range(n_classes):
        rows.append({"Class": class_names[i], "Split": split_name,
                     "Count": int(counts[i]), "Pct": 100.0 * counts[i] / len(y)})

df_dist = pd.DataFrame(rows)
pivot = df_dist.pivot_table(index="Class", columns="Split",
                            values=["Count", "Pct"], sort=False)
pivot = pivot.reindex(columns=[
    ("Count", "Train"), ("Pct", "Train"),
    ("Count", "Val"),   ("Pct", "Val"),
    ("Count", "Test"),  ("Pct", "Test"),
])
pivot.columns = ["Train_n", "Train_%", "Val_n", "Val_%", "Test_n", "Test_%"]
for c in ["Train_n", "Val_n", "Test_n"]:
    pivot[c] = pivot[c].astype(int)

print("Class distribution across splits:")
print(pivot.to_string(float_format="%.1f"))

train_counts = np.bincount(y_train, minlength=n_classes)
nonzero_min = train_counts[train_counts > 0].min()
print(f"\nImbalance ratio (max / min): {train_counts.max() / nonzero_min:,.0f}:1")
print(f"Classes with < 100 training samples: {((train_counts > 0) & (train_counts < 100)).sum()}")

In [ ]:
# ---- 4.2  Class distribution bar chart (linear + log) ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ["#d9534f" if c < 100 else "#5bc0de" for c in train_counts]

bars = axes[0].barh(range(n_classes), train_counts, color=colors)
axes[0].set_yticks(range(n_classes))
axes[0].set_yticklabels(class_names, fontsize=8)
axes[0].set_xlabel("Number of Flows")
axes[0].set_title("Fig 1a -- Training Set Class Distribution")
axes[0].invert_yaxis()
for bar, cnt in zip(bars, train_counts):
    if cnt > 0:
        axes[0].text(bar.get_width() + train_counts.max() * 0.01,
                     bar.get_y() + bar.get_height() / 2,
                     f"{cnt:,}", va="center", fontsize=7)

nonzero = np.maximum(train_counts, 0.5)
axes[1].barh(range(n_classes), nonzero, color=colors)
axes[1].set_xscale("log")
axes[1].set_yticks(range(n_classes))
axes[1].set_yticklabels(class_names, fontsize=8)
axes[1].set_xlabel("Number of Flows (log)")
axes[1].set_title("Fig 1b -- Class Distribution (Log Scale)")
axes[1].invert_yaxis()

fig.suptitle("Red = fewer than 100 training samples", fontsize=9, y=0.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png")
plt.show()

In [ ]:
# ---- 4.3  Data quality audit ----
for name, X in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    n_nan  = np.isnan(X).sum()
    n_inf  = np.isinf(X).sum()
    n_clip = (np.abs(X) > clip_value).sum()
    zv     = (X.std(axis=0) < 1e-8).sum()
    flag   = "  CLEAN" if (n_nan + n_inf) == 0 else "  ** ISSUES **"
    print(f"{name:5s}  NaN={n_nan:,}  Inf={n_inf:,}  "
          f"Beyond|{clip_value}|={n_clip:,}  ZeroVar={zv}{flag}")

In [ ]:
# ---- 4.4  Feature statistics (top / bottom by variance) ----
stats = pd.DataFrame({
    "Feature": feature_names,
    "Mean":     X_train.mean(axis=0),
    "Std":      X_train.std(axis=0),
    "Min":      X_train.min(axis=0),
    "Max":      X_train.max(axis=0),
    "Skew":     pd.Series([float(np.mean(((X_train[:, i] - X_train[:, i].mean()) / max(X_train[:, i].std(), 1e-8))**3))
                           for i in range(n_features)]),
    "Kurtosis": pd.Series([float(np.mean(((X_train[:, i] - X_train[:, i].mean()) / max(X_train[:, i].std(), 1e-8))**4) - 3)
                           for i in range(n_features)]),
})

print("Top 10 features by variance:")
print(stats.nlargest(10, "Std").to_string(index=False, float_format="%.3f"))

print("\nBottom 10 features by variance (near-constant):")
print(stats.nsmallest(10, "Std").to_string(index=False, float_format="%.3f"))

highly_skewed = stats[stats["Skew"].abs() > 2]
print(f"\nHighly skewed features (|skew| > 2): {len(highly_skewed)} / {n_features}")

In [ ]:
# ---- 4.5  Feature histograms (top 12 by variance) ----
variances = X_train.var(axis=0)
top_idx = np.argsort(variances)[::-1][:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, idx in zip(axes.ravel(), top_idx):
    ax.hist(X_train[:, idx], bins=80, alpha=0.75, edgecolor="white", linewidth=0.5)
    ax.set_title(feature_names[idx], fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle("Fig 2 -- Feature Distributions (Top 12 by Variance)", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_histograms.png")
plt.show()

print("\nTop 12 features by variance (shown in Fig 2):")
print(f"{'Feature':>35s}  {'Variance':>10s}  {'Mean':>8s}  {'Std':>8s}")
print("-" * 70)
for idx in top_idx:
    print(f"{feature_names[idx]:>35s}  {variances[idx]:>10.4f}  {X_train[:, idx].mean():>8.4f}  {X_train[:, idx].std():>8.4f}")


In [ ]:
# ---- 4.6  Correlation heatmap ----
rng = np.random.default_rng(SEED)
sub_idx = rng.choice(len(X_train), size=min(50_000, len(X_train)), replace=False)
corr = np.corrcoef(X_train[sub_idx].T)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
tick_idx = list(range(0, n_features, 5))
ax.set_xticks(tick_idx)
ax.set_yticks(tick_idx)
ax.set_xticklabels([feature_names[i][:15] for i in tick_idx], rotation=90, fontsize=6)
ax.set_yticklabels([feature_names[i][:15] for i in tick_idx], fontsize=6)
ax.set_title("Fig 3 -- Feature Correlation Matrix")
plt.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_matrix.png")
plt.show()

upper = np.triu(np.abs(corr), k=1)
high_corr = np.argwhere(upper > 0.95)
print(f"Highly correlated pairs (|r| > 0.95): {len(high_corr)}")
for i, j in high_corr[:10]:
    print(f"  {feature_names[i]:>35s} <-> {feature_names[j]:<35s}  r={corr[i,j]:.3f}")
if len(high_corr) > 10:
    print(f"  ... and {len(high_corr) - 10} more")
print("\nNote: redundant features are kept because feature count must match")
print("the C++ NativeFlowExtractor (77 features).")

<a id="5"></a>

---
## 5. Model Architecture

**CNN-BiLSTM hybrid** -- a pragmatic choice for this project:

| Block | Purpose | Why |
|-------|---------|-----|
| 2x Conv1D | Local pattern extraction | Captures feature-group interactions (port + protocol, flag clusters, IAT patterns) |
| BiLSTM | Cross-feature dependencies | Treats features as an ordered sequence to model long-range interactions bidirectionally |
| Dense head | Classification | BatchNorm + Dropout for regularisation |

**Limitations / honest assessment:**
- Treating non-sequential features as a sequence is debatable. An MLP or
  tree-based model (XGBoost, LightGBM) may perform comparably or better.
- No hyperparameter search was performed; these values come from the initial
  implementation and have not been ablated.

```
Input (batch, 77)
  -> unsqueeze -> (batch, 1, 77)
  -> Conv1d(1, 64, k=3, pad=1) -> BN -> ReLU -> Drop(0.2)
  -> Conv1d(64, 128, k=3, pad=1) -> BN -> ReLU -> Drop(0.2)
  -> MaxPool1d(2) -> (batch, 128, 38)
  -> permute -> (batch, 38, 128)
  -> BiLSTM(128, hidden=128) -> last step -> (batch, 256)
  -> Drop(0.3)
  -> Linear(256, 256) -> BN -> ReLU -> Drop(0.4)
  -> Linear(256, 128) -> BN -> ReLU -> Drop(0.3)
  -> Linear(128, 16) -> [softmax added at ONNX export]
```

In [ ]:
class CnnBiLstm(nn.Module):
    """CNN-BiLSTM hybrid for network intrusion detection.

    Parameters
    ----------
    n_features : int
        Number of input features (77).
    n_classes : int
        Number of output classes (16).
    cnn_channels : tuple[int, int]
        Output channels for the two Conv1d layers.
    kernel_size : int
        Conv1d kernel width.
    lstm_hidden : int
        BiLSTM hidden size (output = 2 * lstm_hidden).
    dense_widths : tuple[int, int]
        Widths of the two FC layers before the logit head.
    drop_cnn : float
        Dropout rate in CNN blocks.
    drop_lstm : float
        Dropout rate after BiLSTM.
    drop_dense : tuple[float, float]
        Dropout rates in the dense block.
    """

    def __init__(
        self,
        n_features: int,
        n_classes: int,
        *,
        cnn_channels: tuple = CNN_CHANNELS,
        kernel_size: int = CNN_KERNEL,
        lstm_hidden: int = LSTM_HIDDEN,
        dense_widths: tuple = DENSE_WIDTHS,
        drop_cnn: float = DROP_CNN,
        drop_lstm: float = DROP_LSTM,
        drop_dense: tuple = DROP_DENSE,
    ):
        super().__init__()
        c1, c2 = cnn_channels
        d1, d2 = dense_widths
        dd1, dd2 = drop_dense

        self.cnn = nn.Sequential(
            nn.Conv1d(1, c1, kernel_size=kernel_size, padding=1),
            nn.BatchNorm1d(c1),
            nn.ReLU(),
            nn.Dropout(drop_cnn),
            nn.Conv1d(c1, c2, kernel_size=kernel_size, padding=1),
            nn.BatchNorm1d(c2),
            nn.ReLU(),
            nn.Dropout(drop_cnn),
            nn.MaxPool1d(kernel_size=2),
        )

        self.lstm = nn.LSTM(
            input_size=c2,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_dropout = nn.Dropout(drop_lstm)

        self.classifier = nn.Sequential(
            nn.Linear(2 * lstm_hidden, d1),
            nn.BatchNorm1d(d1),
            nn.ReLU(),
            nn.Dropout(dd1),
            nn.Linear(d1, d2),
            nn.BatchNorm1d(d2),
            nn.ReLU(),
            nn.Dropout(dd2),
            nn.Linear(d2, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)          # (batch, 77) -> (batch, 1, 77)
        x = self.cnn(x)             # -> (batch, C2, 38)
        x = x.permute(0, 2, 1)     # -> (batch, 38, C2)
        x, _ = self.lstm(x)         # -> (batch, 38, 2*H)
        x = x[:, -1, :]            # last step -> (batch, 2*H)
        x = self.lstm_dropout(x)
        return self.classifier(x)   # -> (batch, n_classes)

In [ ]:
# ---- Instantiate + sanity check ----
model = CnnBiLstm(n_features, n_classes).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params:     {total_params:>10,}")
print(f"Trainable params: {trainable_params:>10,}")
print(f"Estimated size:   {total_params * 4 / 1e6:.1f} MB (fp32)")
print()

# Forward pass sanity check
dummy = torch.randn(4, n_features, device=device)
with torch.no_grad():
    out = model(dummy)
assert out.shape == (4, n_classes), f"Expected (4, {n_classes}), got {out.shape}"
print(f"Forward pass: {dummy.shape} -> {out.shape}  OK")
print()
print(model)

<a id="6"></a>

---
## 6. DataLoaders & Weighted Sampling

The LSNM2024 dataset is **heavily imbalanced**. Without correction the model
would simply predict the majority class.

**Strategy:** `WeightedRandomSampler` with inverse-frequency weights **capped at
100x the median weight**. This ensures ultra-rare classes are upsampled
significantly but not absurdly. No class weights in the loss (avoids double
compensation).

In [ ]:
# ---- Tensors ----
train_dataset = TensorDataset(torch.from_numpy(X_train).float(),
                              torch.from_numpy(y_train).long())
val_dataset   = TensorDataset(torch.from_numpy(X_val).float(),
                              torch.from_numpy(y_val).long())
test_dataset  = TensorDataset(torch.from_numpy(X_test).float(),
                              torch.from_numpy(y_test).long())

# ---- WeightedRandomSampler ----
class_counts = np.bincount(y_train, minlength=n_classes)
class_weights = np.zeros(n_classes, dtype=np.float64)
for i in range(n_classes):
    if class_counts[i] > 0:
        class_weights[i] = 1.0 / class_counts[i]

nonzero_w = class_weights[class_weights > 0]
if len(nonzero_w) > 0:
    median_w = np.median(nonzero_w)
    cap = SAMPLER_CAP_RATIO * median_w
    class_weights = np.minimum(class_weights, cap)

sample_weights = class_weights[y_train]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(y_train),
    replacement=True,
)

# ---- DataLoaders ----
NUM_WORKERS = 0 if _COLAB else min(4, os.cpu_count() or 1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"{'Class':>30s}  {'Count':>9s}  {'Weight':>12s}  {'Capped?':>7s}")
print("-" * 65)
for i in range(n_classes):
    raw = 1.0 / class_counts[i] if class_counts[i] > 0 else 0.0
    capped = "yes" if class_counts[i] > 0 and class_weights[i] < raw else ""
    print(f"{class_names[i]:>30s}  {class_counts[i]:>9,}  {class_weights[i]:>12.6e}  {capped:>7s}")

print(f"\nTrain batches: {len(train_loader):,}  |  Val: {len(val_loader):,}  |  Test: {len(test_loader):,}")

# ---- Quick sampler balance check ----
check_labels = []
for bi, (_, yb) in enumerate(train_loader):
    check_labels.append(yb.numpy())
    if bi >= 19:
        break
check_labels = np.concatenate(check_labels)
check_counts = np.bincount(check_labels, minlength=n_classes)
print(f"Sampler check ({len(check_labels):,} samples):  "
      f"{(check_counts > 0).sum()}/{n_classes} classes seen  "
      f"ratio={check_counts.max() / max(check_counts[check_counts > 0].min(), 1):.1f}x")

<a id="7"></a>

---
## 7. Training

| Choice | Why |
|--------|-----|
| `CrossEntropyLoss(label_smoothing=0.05)` | Reduced from 0.1 to preserve recall on rare classes. **No** class weights -- sampler handles imbalance. |
| `AdamW` | Decoupled weight decay (Loshchilov & Hutter, 2019). |
| `ReduceLROnPlateau(factor=0.5, patience=3)` | Halves LR when val loss plateaus. |
| Gradient clipping `max_norm=1.0` | BiLSTM gradients can explode. |
| **AMP (mixed precision)** | ~2x throughput on T4/V100/A100. Disabled on CPU. |
| Early stopping `patience=10` | Best checkpoint saved; wider window for convergence. |
| **Val macro-F1 tracked** | Accuracy is misleading for imbalanced data. |

In [ ]:
# ---- Loss, optimiser, scheduler ----
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE,
)
scaler = torch.amp.GradScaler(enabled=USE_AMP)

print(f"Loss:      CrossEntropyLoss(label_smoothing={LABEL_SMOOTHING})")
print(f"Optimiser: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler: ReduceLROnPlateau(factor={LR_FACTOR}, patience={LR_PATIENCE})")
print(f"Grad clip: {GRAD_CLIP_NORM}")
print(f"AMP:       {'enabled' if USE_AMP else 'disabled (CPU)'}")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch with optional AMP.

    Returns
    -------
    avg_loss : float
    accuracy : float
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in tqdm(loader, desc="  train", leave=False):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model. Returns (avg_loss, accuracy, macro_f1)."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for xb, yb in tqdm(loader, desc="  eval ", leave=False):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(xb)
            loss = criterion(logits, yb)

        running_loss += loss.item() * xb.size(0)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(yb.cpu())

    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    total = len(labels)
    acc = (preds == labels).mean()
    mf1 = f1_score(labels, preds, average="macro", zero_division=0)

    return running_loss / total, float(acc), float(mf1)


@torch.no_grad()
def get_predictions(model, loader, device):
    """Collect predictions + softmax probabilities for analysis.

    Returns
    -------
    y_pred : ndarray
    y_true : ndarray
    y_probs : ndarray  (softmax probabilities)
    """
    preds, labels, probs = [], [], []
    sm = nn.Softmax(dim=1)
    for xb, yb in tqdm(loader, desc="predict", leave=False):
        xb = xb.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(xb)
        p = sm(logits.float())
        preds.append(logits.argmax(1).cpu().numpy())
        labels.append(yb.numpy())
        probs.append(p.cpu().numpy())
    return np.concatenate(preds), np.concatenate(labels), np.concatenate(probs)


print("Training functions defined.")

In [ ]:
%%time
# ============== Training loop ==============
best_val_loss = float("inf")
best_val_f1   = 0.0
best_epoch    = 0
patience_ctr  = 0

history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_f1": [],
    "lr": [],
}

CKPT_PATH = OUTPUT_DIR / "best_model.pt"

print(f"Training for up to {MAX_EPOCHS} epochs (patience={PATIENCE})")
print(f"{'Ep':>4s}  {'TrLoss':>8s} {'TrAcc':>7s}  {'VaLoss':>8s} {'VaAcc':>7s} {'VaF1':>7s}  {'LR':>9s}  {'Time':>5s}")
print("-" * 72)

t_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    t_ep = time.time()

    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    va_loss, va_acc, va_f1 = evaluate(model, val_loader, criterion, device)
    scheduler.step(va_loss)

    lr = optimizer.param_groups[0]["lr"]
    dt = time.time() - t_ep

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    history["val_f1"].append(va_f1)
    history["lr"].append(lr)

    improved = va_loss < best_val_loss
    marker = "*" if improved else " "

    print(f"{epoch:>3d}{marker}  {tr_loss:>8.4f} {tr_acc:>7.4f}  "
          f"{va_loss:>8.4f} {va_acc:>7.4f} {va_f1:>7.4f}  "
          f"{lr:>9.2e}  {dt:>4.0f}s")

    if improved:
        best_val_loss = va_loss
        best_val_f1   = va_f1
        best_epoch    = epoch
        patience_ctr  = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": va_loss,
            "val_acc": va_acc,
            "val_f1": va_f1,
            "n_features": n_features,
            "n_classes": n_classes,
            "lstm_hidden": LSTM_HIDDEN,
        }, CKPT_PATH)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break

total_time = time.time() - t_start
print(f"\nDone in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Best epoch {best_epoch}: val_loss={best_val_loss:.4f}  val_f1={best_val_f1:.4f}")

if device.type == "cuda":
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"Peak GPU memory: {peak_mb:.0f} MB")

<a id="8"></a>

---
## 8. Training Curves

Check for:
- **Convergence** -- both train/val curves should flatten.
- **Overfitting** -- train loss keeps dropping while val loss rises.
- **LR schedule** -- step-downs should correlate with val-loss plateaus.
- **F1 trend** -- macro F1 on validation should steadily increase.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ep = range(1, len(history["train_loss"]) + 1)

axes[0, 0].plot(ep, history["train_loss"], label="Train", lw=2)
axes[0, 0].plot(ep, history["val_loss"], label="Val", lw=2)
axes[0, 0].axvline(best_epoch, color="r", ls="--", alpha=0.5, label=f"Best (ep {best_epoch})")
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Fig 4a -- Loss"); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(ep, history["train_acc"], label="Train", lw=2)
axes[0, 1].plot(ep, history["val_acc"], label="Val", lw=2)
axes[0, 1].axvline(best_epoch, color="r", ls="--", alpha=0.5)
axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].set_title("Fig 4b -- Accuracy"); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(ep, history["val_f1"], color="purple", lw=2)
axes[1, 0].axvline(best_epoch, color="r", ls="--", alpha=0.5)
axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("Macro F1")
axes[1, 0].set_title("Fig 4c -- Validation Macro F1"); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(ep, history["lr"], color="green", lw=2)
axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("LR")
axes[1, 1].set_title("Fig 4d -- Learning Rate"); axes[1, 1].set_yscale("log")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png")
plt.show()

# ---- Print history data (for text-based analysis) ----
print("\nTraining history (all epochs):")
print(f"{'Ep':>4s}  {'TrLoss':>8s} {'TrAcc':>7s}  {'VaLoss':>8s} {'VaAcc':>7s} {'VaF1':>7s}  {'LR':>9s}")
print("-" * 65)
for e in range(len(history["train_loss"])):
    m = "*" if (e + 1) == best_epoch else " "
    print(f"{e+1:>3d}{m}  {history['train_loss'][e]:>8.4f} {history['train_acc'][e]:>7.4f}  "
          f"{history['val_loss'][e]:>8.4f} {history['val_acc'][e]:>7.4f} {history['val_f1'][e]:>7.4f}  "
          f"{history['lr'][e]:>9.2e}")


<a id="9"></a>

---
## 9. Evaluation on Test Set

Load the **best checkpoint** (lowest validation loss) and evaluate on the
held-out test set:

- 9.1 Test-set loss, accuracy, macro F1
- 9.2 Classification report (per-class P / R / F1)
- 9.3 **Binary detection** -- benign vs. any-attack (operationally most important)
- 9.4 Per-class F1 bar chart
- 9.5 Confusion matrix (counts + normalised)
- 9.6 ROC curves (one-vs-rest)
- 9.7 Prediction confidence analysis
- 9.8 Per-class error analysis

In [ ]:
# ---- Load best checkpoint + compute test metrics ----
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
print(f"  val_loss={ckpt['val_loss']:.4f}  val_acc={ckpt['val_acc']:.4f}  val_f1={ckpt['val_f1']:.4f}")

test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
print(f"\nTest loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}  ({test_acc * 100:.2f}%)")
print(f"Test macro F1: {test_f1:.4f}")

In [ ]:
# ---- Collect predictions + softmax probabilities ----
y_pred, y_true, y_probs = get_predictions(model, test_loader, device)
all_labels = list(range(n_classes))
print(f"Predictions collected: {len(y_pred):,} samples")

In [ ]:
# ---- 9.2  Classification report ----
report = classification_report(
    y_true, y_pred, labels=all_labels, target_names=class_names,
    digits=4, zero_division=0,
)
print(report)
(OUTPUT_DIR / "classification_report.txt").write_text(report)

In [ ]:
# ---- 9.3  Binary detection: benign vs any-attack ----
# False Negative = attack classified as benign -> security breach
# False Positive = benign classified as attack -> alert fatigue

y_binary_true = (y_true > 0).astype(int)
y_binary_pred = (y_pred > 0).astype(int)

binary_report = classification_report(
    y_binary_true, y_binary_pred,
    target_names=["Benign", "Attack (any)"],
    digits=4, zero_division=0,
)
print("Binary Detection (Benign vs Any Attack):")
print(binary_report)

fn = ((y_binary_true == 1) & (y_binary_pred == 0)).sum()
fp = ((y_binary_true == 0) & (y_binary_pred == 1)).sum()
print(f"False negatives (attacks missed):     {fn:>8,}")
print(f"False positives (benign flagged):     {fp:>8,}")
print(f"Attack recall (detection rate):       {recall_score(y_binary_true, y_binary_pred):.4f}")
print(f"Attack precision:                     {precision_score(y_binary_true, y_binary_pred):.4f}")

In [ ]:
# ---- 9.4  Per-class F1 bar chart ----
per_f1 = f1_score(y_true, y_pred, labels=all_labels, average=None, zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#d9534f" if f < 0.5 else "#f0ad4e" if f < 0.8 else "#5cb85c" for f in per_f1]
bars = ax.barh(range(n_classes), per_f1, color=colors)
ax.set_yticks(range(n_classes))
ax.set_yticklabels(class_names, fontsize=9)
ax.set_xlabel("F1-Score")
ax.set_title("Fig 5 -- Per-Class F1-Score on Test Set")
ax.set_xlim(0, 1.1)
ax.invert_yaxis()

for bar, f in zip(bars, per_f1):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{f:.3f}", va="center", fontsize=8)

ax.axvline(0.8, color="gray", ls="--", alpha=0.5, label="F1 = 0.8")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_class_f1.png")
plt.show()

print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print(f"\n{'Class':>30s}  {'F1':>7s}  {'Grade':>5s}")
print("-" * 48)
for ci in range(n_classes):
    grade = "BAD" if per_f1[ci] < 0.5 else "WEAK" if per_f1[ci] < 0.8 else "GOOD"
    print(f"{class_names[ci]:>30s}  {per_f1[ci]:>7.4f}  {grade:>5s}")


In [ ]:
# ---- 9.5  Confusion matrix (counts + normalised) ----
cm = confusion_matrix(y_true, y_pred, labels=all_labels)
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm.astype(float), row_sums,
                    out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], annot_kws={"size": 7})
axes[0].set_title("Fig 6a -- Confusion Matrix (Counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].tick_params(axis="x", rotation=45)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], annot_kws={"size": 7})
axes[1].set_title("Fig 6b -- Confusion Matrix (Recall-normalised)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png")
plt.show()

# ---- Print confusion matrix as text ----
print("\nConfusion matrix (rows=true, cols=predicted):")
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df.to_string())

print("\nRecall-normalised confusion matrix:")
cm_norm_df = pd.DataFrame(np.round(cm_norm, 3), index=class_names, columns=class_names)
print(cm_norm_df.to_string())


In [ ]:
# ---- 9.6  ROC curves (one-vs-rest) ----
y_true_oh = np.eye(n_classes)[y_true]

fig, ax = plt.subplots(figsize=(12, 10))
per_auc = {}
for i in range(n_classes):
    if y_true_oh[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_true_oh[:, i], y_probs[:, i])
    auc_val = roc_auc_score(y_true_oh[:, i], y_probs[:, i])
    per_auc[class_names[i]] = auc_val
    ax.plot(fpr, tpr, label=f"{class_names[i]} (AUC={auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random (0.5)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Fig 7 -- ROC Curves (One-vs-Rest)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png")
plt.show()

macro_auc = np.mean(list(per_auc.values())) if per_auc else 0.0
print(f"Macro AUC-ROC: {macro_auc:.4f}")
print(f"\n{'Class':>30s}  {'AUC':>7s}")
print("-" * 42)
for name, auc_val in sorted(per_auc.items(), key=lambda x: x[1]):
    print(f"{name:>30s}  {auc_val:>7.4f}")


In [ ]:
# ---- 9.7  Prediction confidence analysis ----
max_probs = y_probs.max(axis=1)
correct = y_pred == y_true

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(max_probs[correct], bins=50, alpha=0.7, label="Correct", color="#5cb85c")
axes[0].hist(max_probs[~correct], bins=50, alpha=0.7, label="Incorrect", color="#d9534f")
axes[0].set_xlabel("Max Softmax Probability")
axes[0].set_ylabel("Count")
axes[0].set_title("Fig 8a -- Confidence Distribution")
axes[0].legend()

n_ok  = correct.sum()
n_err = (~correct).sum()
axes[1].axis("off")
txt = (
    f"Total:     {len(y_pred):>9,}\n"
    f"Correct:   {n_ok:>9,}  ({100*n_ok/len(y_pred):.1f}%)\n"
    f"Incorrect: {n_err:>9,}  ({100*n_err/len(y_pred):.1f}%)\n"
    f"\nMean conf (correct):   {(max_probs[correct].mean() if n_ok > 0 else 0.0):.4f}\n"
    f"Mean conf (incorrect): {(max_probs[~correct].mean() if n_err > 0 else 0.0):.4f}\n"
    f"\nHigh-conf errors (>0.9): {(max_probs[~correct] > 0.9).sum():,}\n"
    f"Low-conf correct (<0.5): {(max_probs[correct] < 0.5).sum():,}\n"
    f"\nNote: softmax scores are NOT calibrated\n"
    f"probabilities. A reliability diagram would\n"
    f"be needed to assess calibration quality."
)
axes[1].text(0.1, 0.5, txt, transform=axes[1].transAxes,
             fontsize=10, va="center", fontfamily="monospace")
axes[1].set_title("Fig 8b -- Confidence Summary")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confidence_analysis.png")
plt.show()

# ---- Print confidence stats as text ----
print("\nPrediction confidence analysis:")
print(f"Total:     {len(y_pred):>9,}")
print(f"Correct:   {n_ok:>9,}  ({100*n_ok/len(y_pred):.1f}%)")
print(f"Incorrect: {n_err:>9,}  ({100*n_err/len(y_pred):.1f}%)")
print(f"Mean conf (correct):   {(max_probs[correct].mean() if n_ok > 0 else 0.0):.4f}")
print(f"Mean conf (incorrect): {(max_probs[~correct].mean() if n_err > 0 else 0.0):.4f}")
print(f"High-conf errors (>0.9): {(max_probs[~correct] > 0.9).sum():,}")
print(f"Low-conf correct (<0.5): {(max_probs[correct] < 0.5).sum():,}")


In [ ]:
# ---- 9.8  Per-class error analysis ----
print("=" * 85)
print("Per-class error analysis")
print("=" * 85)

print("\nATTACKS MISCLASSIFIED AS BENIGN (false negatives -- security risk):")
print(f"{'True Class':>30s}  {'As Benign':>9s}  {'Total':>7s}  {'FN Rate':>7s}")
print("-" * 60)
for i in range(1, n_classes):
    mask_true = y_true == i
    if mask_true.sum() == 0:
        continue
    fn = ((y_true == i) & (y_pred == 0)).sum()
    total_i = mask_true.sum()
    rate = fn / total_i if total_i > 0 else 0
    flag = "  !!" if rate > 0.1 else ""
    print(f"{class_names[i]:>30s}  {fn:>9,}  {total_i:>7,}  {rate:>6.2%}{flag}")

print(f"\nTOP CONFUSIONS PER CLASS:")
print(f"{'True Class':>30s}  {'Predicted As':>30s}  {'Count':>6s}  {'% Errors':>8s}")
print("-" * 80)
for i in range(n_classes):
    mask = y_true == i
    if mask.sum() == 0:
        continue
    wrong = mask & (y_pred != i)
    n_err = wrong.sum()
    if n_err == 0:
        print(f"{class_names[i]:>30s}  {'(perfect)':>30s}")
        continue
    err_counts = np.bincount(y_pred[wrong], minlength=n_classes)
    top3 = np.argsort(err_counts)[::-1][:3]
    for rank, j in enumerate(top3):
        if err_counts[j] == 0:
            break
        pct = 100.0 * err_counts[j] / n_err
        name = class_names[i] if rank == 0 else ""
        print(f"{name:>30s}  {class_names[j]:>30s}  {err_counts[j]:>6,}  {pct:>7.1f}%")

<a id="10"></a>

---
## 10. ONNX Export

The C++ runtime (`OnnxAnalyzer`) expects:
- **Input**: `float32 (batch, 77)` -- flat feature vector
- **Output**: `float32 (batch, 16)` -- **softmax probabilities**

Since the training model outputs raw logits, we wrap it with a Softmax layer.
After export we verify:
1. ONNX structural validity (`onnx.checker`)
2. **Numerical equivalence** between PyTorch and ONNX Runtime
3. Inference speed benchmark

In [ ]:
class CnnBiLstmExport(nn.Module):
    """Wraps CnnBiLstm with Softmax for ONNX deployment.

    Training uses raw logits (CrossEntropyLoss includes log-softmax).
    C++ inference needs probabilities directly.
    """

    def __init__(self, model: CnnBiLstm):
        super().__init__()
        self.model = model
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.softmax(self.model(x))

In [ ]:
# ---- Export to ONNX ----
model_cpu = copy.deepcopy(model).cpu().eval()

# Flatten LSTM parameters for contiguous memory layout (required by ONNX tracer)
if hasattr(model_cpu, "lstm"):
    model_cpu.lstm.flatten_parameters()

export_model = CnnBiLstmExport(model_cpu)
export_model.eval()

ONNX_PATH = str(OUTPUT_DIR / "model.onnx")
OPSET = 17
dummy = torch.randn(1, n_features)

print(f"Exporting ONNX (opset {OPSET})...")
with torch.no_grad():
    torch.onnx.export(
        export_model, (dummy,), ONNX_PATH,
        opset_version=OPSET,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        do_constant_folding=True,
    )

onnx_p = Path(ONNX_PATH)
print(f"  {onnx_p.name}: {onnx_p.stat().st_size / 1e6:.2f} MB")
data_p = Path(ONNX_PATH + ".data")
if data_p.exists():
    print(f"  {data_p.name}: {data_p.stat().st_size / 1e6:.2f} MB")

# ---- Structural validation ----
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("Structural validation: PASSED")

sess = ort.InferenceSession(ONNX_PATH)
inp  = sess.get_inputs()[0]
outp = sess.get_outputs()[0]
print(f"Input:  name={inp.name!r}  shape={inp.shape}  type={inp.type}")
print(f"Output: name={outp.name!r}  shape={outp.shape}  type={outp.type}")

# ---- Numerical equivalence: PyTorch vs ONNX Runtime ----
test_inputs = np.random.randn(32, n_features).astype(np.float32)

with torch.no_grad():
    pt_out = export_model(torch.from_numpy(test_inputs)).numpy()

ort_out = sess.run([outp.name], {inp.name: test_inputs})[0]

max_diff = np.abs(pt_out - ort_out).max()
mean_diff = np.abs(pt_out - ort_out).mean()
print(f"\nMax diff:  {max_diff:.8f}")
print(f"Mean diff: {mean_diff:.8f}")
assert max_diff < 1e-4, f"ONNX output diverges from PyTorch! max_diff={max_diff}"
print("Numerical equivalence: PASSED")

softmax_sums = ort_out.sum(axis=1)
print(f"Softmax sum range: [{softmax_sums.min():.6f}, {softmax_sums.max():.6f}]  (expect ~1.0)")


In [ ]:
# ---- ONNX Runtime inference benchmark ----
print(f"Provider: {sess.get_providers()}")

single = np.random.randn(1, n_features).astype(np.float32)
for _ in range(10):
    sess.run([outp.name], {inp.name: single})

N = 100
t0 = time.time()
for _ in range(N):
    sess.run([outp.name], {inp.name: single})
lat_ms = (time.time() - t0) / N * 1000

batch = np.random.randn(256, n_features).astype(np.float32)
for _ in range(10):
    sess.run([outp.name], {inp.name: batch})

t0 = time.time()
for _ in range(N):
    sess.run([outp.name], {inp.name: batch})
batch_ms = (time.time() - t0) / N * 1000
throughput = 256 / (batch_ms / 1000)

print(f"Single-sample latency:  {lat_ms:.2f} ms")
print(f"Batch(256) latency:     {batch_ms:.2f} ms")
print(f"Throughput:             {throughput:,.0f} samples/sec")

<a id="11"></a>

---
## 11. Summary & Artifacts

In [ ]:
# ---- Copy metadata alongside ONNX model ----
shutil.copy2(DATA_DIR / "model_metadata.json", OUTPUT_DIR / "model_metadata.json")

# ---- Training summary ----
summary = {
    "best_epoch": best_epoch,
    "best_val_loss": round(best_val_loss, 6),
    "best_val_f1": round(best_val_f1, 6),
    "test_loss": round(test_loss, 6),
    "test_accuracy": round(test_acc, 6),
    "test_macro_f1": round(float(macro_f1), 6),
    "test_weighted_f1": round(float(weighted_f1), 6),
    "test_macro_auc": round(float(macro_auc), 6),
    "total_params": total_params,
    "n_features": n_features,
    "n_classes": n_classes,
    "hyperparameters": {
        "cnn_channels": list(CNN_CHANNELS),
        "cnn_kernel": CNN_KERNEL,
        "lstm_hidden": LSTM_HIDDEN,
        "dense_widths": list(DENSE_WIDTHS),
        "drop_cnn": DROP_CNN,
        "drop_lstm": DROP_LSTM,
        "drop_dense": list(DROP_DENSE),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "lr_factor": LR_FACTOR,
        "lr_patience": LR_PATIENCE,
        "grad_clip_norm": GRAD_CLIP_NORM,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "sampler_cap_ratio": SAMPLER_CAP_RATIO,
        "use_amp": USE_AMP,
    },
    "training_time_sec": round(total_time, 1),
    "device": str(device),
    "seed": SEED,
    "pytorch_version": torch.__version__,
}

with open(OUTPUT_DIR / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

# ---- List output artifacts ----
print(f"\n{'File':45s}  {'Size':>10s}  {'SHA256 (first 12)':>14s}")
print("-" * 75)
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        h = hashlib.sha256(p.read_bytes()).hexdigest()[:12]
        print(f"{p.name:45s}  {size_mb:>8.2f} MB  {h}")

print("\nTo deploy:")
print("  1. Copy model.onnx (+ model.onnx.data if present) to models/")
print("  2. Copy model_metadata.json to models/")
print("  3. Rebuild the C++ project")

# ---- Flush and close the Tee logger ----
if isinstance(sys.stdout, _Tee):
    sys.stdout.flush()
    sys.stdout.close()
    sys.stdout = sys.__stdout__
if isinstance(sys.stderr, _Tee):
    sys.stderr.flush()
    sys.stderr.close()
    sys.stderr = sys.__stderr__

print(f"\nTraining log saved to: {_LOG_PATH.resolve()}")
print(f"Download it from the Colab file browser or via:")
print(f"  from google.colab import files; files.download('{_LOG_PATH}')")

---
## Conclusions & Limitations

### Key findings

- The CNN-BiLSTM model achieves the metrics shown above on the LSNM2024 test set.
- Binary detection (benign vs. any attack) performance is reported in Sec 9.3 --
  this is the operationally most important metric.
- Ultra-rare classes (DDoS-ICMP, ICMP-Flood, DDoS-RawIP) have limited training
  data even after max-flow splitting and may show unreliable F1 scores.

### Known limitations

1. **No baseline comparison.** We did not benchmark against simpler models
   (logistic regression, random forest, MLP). The CNN-BiLSTM's additional
   complexity may not be justified.
2. **No hyperparameter search.** All values are from the initial implementation.
   Optuna / Ray Tune should be used for systematic tuning.
3. **Single dataset, single split.** Results may not generalise. Cross-validation
   or bootstrap confidence intervals would strengthen claims.
4. **Treating features as a sequence is debatable.** Flow features are not
   inherently sequential; the BiLSTM may not provide benefit over a pure MLP.
5. **Softmax scores are not calibrated.** High-confidence predictions do not
   necessarily correspond to high true probability.
6. **No concept-drift analysis.** Real-world traffic patterns change over time.

### Future work

- Benchmark against XGBoost / LightGBM baseline
- Hyperparameter search with Optuna
- Temperature scaling for confidence calibration
- Data augmentation for ultra-rare classes
- Experiment tracking with Weights & Biases or MLflow